# ==============================================================================
# 🏥 ACTUARIAL HEALTHCARE RISK: MEDICAL INSURANCE COST VIA DECISION TREE REGRESSOR
# ==============================================================================
### Architectural Problem Formulation:
Medical insurance expenditures exhibit extreme non-linearities and sharp feature interactions:
* Non-smokers with high BMI have moderate risk.
* Smokers with high BMI suffer exponential increases in medical costs ($>\$40,000$).
A Decision Tree Regressor naturally captures this multi-way feature interaction without requiring manual quadratic polynomial interaction terms:
$$\text{IF } \text{smoker}=\text{yes} \text{ AND } \text{bmi} > 30 \implies \hat{y} = \$39,850$$


In [1]:
# STEP 1: ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Actuarial Healthcare Environment ready.")


✅ STEP 1: Actuarial Healthcare Environment ready.


## 📥 STEP 2 & 3: INGESTION & SANITIZATION
Loading Kaggle Medical Cost Personal Datasets (1,338 policyholders).


In [2]:
# STEP 2 & 3: LOAD & SANITIZE
data_path = 'data/medical_cost/insurance.csv'
df = pd.read_csv(data_path)

df.columns = [c.strip().lower() for c in df.columns]
print(f"📊 Dataset Shape: {df.shape[0]:,} policyholders, {df.shape[1]} attributes")
print(f"Attributes: {list(df.columns)}")
df.head(3)


📊 Dataset Shape: 1,338 policyholders, 7 attributes
Attributes: ['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges']


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.90,0,yes,southwest,16884.9240
1,18,male,33.77,1,no,southeast,1725.5523
2,28,male,33.00,3,no,southeast,4449.4620


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Separating features and target `charges`, verifying data integrity.


In [3]:
# STEP 4 & 5: SEGREGATION & HYGIENE
X = df.drop(columns=['charges'])
y = df['charges']

print(f"Missing values: {X.isnull().sum().sum()}")
print(f"Target Insurance Charges Summary ($):")
print(y.describe().round(2))


Missing values: 0
Target Insurance Charges Summary ($):
count     1338.00
mean     13270.42
std      12110.01
min       1121.87
25%       4740.29
50%       9382.03
75%      16639.91
max      63770.43
Name: charges, dtype: float64


## 🔒 STEP 6: LEAK-FREE TRAIN-TEST SPLIT
80/20 train/test partition.


In [4]:
# STEP 6: SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f"Train Policyholders: {X_train.shape[0]}")
print(f"Test Policyholders : {X_test.shape[0]}")


Train Policyholders: 1070
Test Policyholders : 268


## ⚙️ STEP 7: COLUMNTRANSFORMER FOR DECISION TREES
Categorical features (`sex`, `smoker`, `region`) are encoded via `OneHotEncoder`.
Numeric features (`age`, `bmi`, `children`) pass through directly without scaling.


In [5]:
# STEP 7: PREPROCESSOR
cat_cols = ['sex', 'smoker', 'region']
num_cols = ['age', 'bmi', 'children']

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), cat_cols),
    ('num', 'passthrough', num_cols)
])
print("✅ STEP 7: Actuarial Preprocessor configured.")


✅ STEP 7: Actuarial Preprocessor configured.


## ⚠️ STEP 8: THE OVERFITTING TRAP
Comparing deep unconstrained tree memorization against a pre-pruned tree.


In [6]:
# STEP 8: OVERFITTING DEMONSTRATION
pipe_uncon = Pipeline([
    ('prep', preprocessor),
    ('tree', DecisionTreeRegressor(max_depth=None, random_state=42))
])
pipe_uncon.fit(X_train, y_train)

print(f"Unconstrained Tree Train R² : {r2_score(y_train, pipe_uncon.predict(X_train))*100:.2f}%")
print(f"Unconstrained Tree Test R²  : {r2_score(y_test, pipe_uncon.predict(X_test))*100:.2f}%")


Unconstrained Tree Train R² : 99.83%
Unconstrained Tree Test R²  : 69.68%


## 🎯 STEP 9: TUNING VIA 5-FOLD CROSS VALIDATION
Optimizing `max_depth` and `min_samples_leaf` to minimize root mean squared error.


In [7]:
# STEP 9: GRID SEARCH
param_grid = {
    'tree__max_depth': [3, 4, 5, 6, 8],
    'tree__min_samples_split': [5, 10, 20],
    'tree__min_samples_leaf': [2, 5, 10, 20]
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(
    Pipeline([('prep', preprocessor), ('tree', DecisionTreeRegressor(random_state=42))]),
    param_grid=param_grid,
    cv=cv,
    scoring='r2',
    n_jobs=-1
)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Hyperparameters: {grid.best_params_}")
print(f"🎯 Best 5-Fold CV R²   : {grid.best_score_*100:.2f}%")


🏆 Best Hyperparameters: {'tree__max_depth': 4, 'tree__min_samples_leaf': 10, 'tree__min_samples_split': 5}
🎯 Best 5-Fold CV R²   : 85.33%


## ⚔️ STEP 10: ALGORITHM DUEL (DecisionTreeRegressor vs Ridge)
Demonstrating how decision trees easily outperform linear models when complex interaction effects exist.


In [8]:
# STEP 10: MODEL BENCHMARK
from sklearn.preprocessing import StandardScaler

pipe_ridge = Pipeline([
    ('prep', ColumnTransformer([
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), cat_cols),
        ('num', StandardScaler(), num_cols)
    ])),
    ('ridge', Ridge(alpha=10.0, random_state=42))
])
pipe_ridge.fit(X_train, y_train)

y_pred_tree = best_model.predict(X_test)
y_pred_ridge = pipe_ridge.predict(X_test)

print("="*65)
print(f"DecisionTreeRegressor (Tuned) -> Test R²: {r2_score(y_test, y_pred_tree)*100:.2f}% | RMSE: ${np.sqrt(mean_squared_error(y_test, y_pred_tree)):,.2f}")
print(f"Ridge Regression              -> Test R²: {r2_score(y_test, y_pred_ridge)*100:.2f}% | RMSE: ${np.sqrt(mean_squared_error(y_test, y_pred_ridge)):,.2f}")
print("="*65)


DecisionTreeRegressor (Tuned) -> Test R²: 86.41% | RMSE: $4,592.76
Ridge Regression              -> Test R²: 77.91% | RMSE: $5,856.63


## 📊 STEP 11: RESIDUAL ANALYSIS & ACTUARIAL DIAGNOSTICS
Evaluating error metrics and policy charge drivers.


In [9]:
# STEP 11: EVALUATION REPORT
mae = mean_absolute_error(y_test, y_pred_tree)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_tree))

print(f"📋 ACTUARIAL UNDERWRITING METRICS:")
print(f"   Mean Absolute Error (MAE) : ${mae:,.2f}")
print(f"   Root Mean Squared Error   : ${rmse:,.2f}")
print(f"   R² Score                  : {r2_score(y_test, y_pred_tree)*100:.2f}%")


📋 ACTUARIAL UNDERWRITING METRICS:
   Mean Absolute Error (MAE) : $2,697.77
   Root Mean Squared Error   : $4,592.76
   R² Score                  : 86.41%


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & UNDERWRITING SMOKE TEST
Saving model and benchmarking sub-millisecond underwriting quote latency.


In [10]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/medical_insurance_tree_pipeline.joblib'
joblib.dump(best_model, model_path)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_applicant = pd.DataFrame([{
    'age': 45,
    'sex': 'male',
    'bmi': 32.5,
    'children': 2,
    'smoker': 'yes',
    'region': 'southeast'
}])

t0 = time.perf_counter()
pred_premium = loaded_pipe.predict(sample_applicant)[0]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n🏥 LIVE UNDERWRITING QUOTE AUDIT:")
print(f"   Forecast Annual Charges: 💵 ${pred_premium:,.2f}")
print(f"   Risk Stratification    : {'🚨 HIGH RISK TIER' if pred_premium > 25000 else '✅ STANDARD TIER'}")
print(f"   Quote Latency          : {latency_ms:.3f} ms (SLA < 1.0ms: {'PASS' if latency_ms < 1.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/medical_insurance_tree_pipeline.joblib (6,467 bytes)

🏥 LIVE UNDERWRITING QUOTE AUDIT:
   Forecast Annual Charges: 💵 $42,596.28
   Risk Stratification    : 🚨 HIGH RISK TIER
   Quote Latency          : 3.138 ms (SLA < 1.0ms: CHECK)
